# Conditional Image Generation (cGAN) on the Fashion-MNIST Dataset

This notebook implements a Conditional Generative Adversarial Network (cGAN) using PyTorch. The model is trained on the Fashion-MNIST dataset, which allows us to generate specific categories of clothing items by passing class labels to both the generator and discriminator.

This project is designed to be lightweight and run efficiently on a standard GPU (like a free Google Colab T4).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

### 1. Define the Generator and Discriminator

The key to a *conditional* GAN is passing the label (the category of clothing) into **both** networks alongside the images and noise. We use `nn.Embedding` to convert the class labels into a continuous vector format, which is then concatenated with the image or noise data.

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim, num_classes, img_dim):
        super(Generator, self).__init__()
        # Embed the label into a vector of the same size as the number of classes
        self.label_emb = nn.Embedding(num_classes, num_classes)
        
        self.model = nn.Sequential(
            # Input = noise vector + label embedding
            nn.Linear(noise_dim + num_classes, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, img_dim),
            nn.Tanh() # Tanh ensures pixel values are between -1 and 1
        )

    def forward(self, noise, labels):
        c = self.label_emb(labels)
        x = torch.cat([noise, c], 1)
        out = self.model(x)
        # Reshape the flat vector back into a 28x28 image
        return out.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self, num_classes, img_dim):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        
        self.model = nn.Sequential(
            # Input = flattened image + label embedding
            nn.Linear(img_dim + num_classes, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid() # Outputs a probability (0 for fake, 1 for real)
        )

    def forward(self, img, labels):
        img_flat = img.view(img.size(0), -1)
        c = self.label_emb(labels)
        x = torch.cat([img_flat, c], 1)
        out = self.model(x)
        return out

### 2. Data Loading and Training Loop

This handles downloading the Fashion-MNIST dataset, setting up the optimizers, and running the adversarial training loop where the two models compete against each other.

In [ ]:
# --- Hyperparameters ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lr = 0.0002
batch_size = 64
noise_dim = 100
epochs = 50
num_classes = 10
img_dim = 28 * 28 * 1

# --- Data Preparation ---
# Normalize images to [-1, 1] to match the Generator's Tanh activation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

dataset = datasets.FashionMNIST(root='dataset/', train=True, transform=transform, download=True)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# --- Initialization ---
gen = Generator(noise_dim, num_classes, img_dim).to(device)
disc = Discriminator(num_classes, img_dim).to(device)

opt_gen = optim.Adam(gen.parameters(), lr=lr)
opt_disc = optim.Adam(disc.parameters(), lr=lr)
criterion = nn.BCELoss()

# --- Training Loop ---
for epoch in range(epochs):
    for batch_idx, (real, labels) in enumerate(loader):
        real = real.to(device)
        labels = labels.to(device)
        current_batch_size = real.shape[0]

        ### 1. Train Discriminator: Maximize log(D(x)) + log(1 - D(G(z))) ###
        # Generate fake images
        noise = torch.randn(current_batch_size, noise_dim).to(device)
        fake = gen(noise, labels)
        
        # Pass real and fake images through discriminator
        disc_real = disc(real, labels)
        lossD_real = criterion(disc_real, torch.ones_like(disc_real))
        
        disc_fake = disc(fake.detach(), labels) # detach to avoid updating generator here
        lossD_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        
        lossD = (lossD_real + lossD_fake) / 2
        disc.zero_grad()
        lossD.backward()
        opt_disc.step()

        ### 2. Train Generator: Maximize log(D(G(z))) ###
        # Evaluate generator's fake images with the updated discriminator
        output = disc(fake, labels)
        lossG = criterion(output, torch.ones_like(output)) # The generator wants the discriminator to think it's real (1)
        
        gen.zero_grad()
        lossG.backward()
        opt_gen.step()

    print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {lossD:.4f}, Loss G: {lossG:.4f}")

### 3. Evaluating the Controlled Output

Once training is complete, you can pass specific class labels to the generator to see if it learned to produce the requested items.

In [ ]:
# Generate specific items (e.g., class 9 = Ankle Boot, class 0 = T-shirt)
gen.eval()
with torch.no_grad():
    # Let's generate 5 ankle boots
    target_labels = torch.tensor([9, 9, 9, 9, 9]).to(device) 
    test_noise = torch.randn(5, noise_dim).to(device)
    
    generated_images = gen(test_noise, target_labels).cpu()
    
    # Plot the results
    fig, axes = plt.subplots(1, 5, figsize=(10, 2))
    for i in range(5):
        axes[i].imshow(generated_images[i].squeeze(), cmap='gray')
        axes[i].axis('off')
        axes[i].set_title("Ankle Boot")
    plt.show()